# PFE ML — Phase E : Calibration des probabilités du gagnant HGB optimisé

Les phases A-D ont établi que le HGB optimisé classe bien les entreprises : AP 0,30 / AUC 0,88 sur 2023. La Phase F a confirmé que la qualité de classement tient sur les secteurs, les cohortes d'âge et les principales formes juridiques. La Phase E répond à une question différente, importante spécifiquement parce que le **front-end Angular expose `probabilite_cessation` aux utilisateurs sous forme de probabilité numérique** :

**Question de la Phase E :** lorsque le modèle sort *p = 0,7*, est-ce qu'environ 70 % de ces entreprises cessent réellement ? Ou le modèle classe-t-il bien mais à une mauvaise échelle ?

La qualité de classement (AUC, AP) et la calibration sont des dimensions indépendantes. Un modèle à AUC élevé peut quand même être mal calibré — par exemple un HGB en `class_weight='balanced'` sur-estime systématiquement la probabilité de la classe minoritaire parce que la fonction de perte sur-pondère les positifs. Que cela importe ou non dépend de la façon dont le score est consommé en aval :
- Si le produit se contente de classer les entreprises par risque → le classement suffit, la calibration n'est pas pertinente.
- Si le produit affiche *« cette entreprise a 73 % de chances de cesser dans les 12 prochains mois »* → la calibration est essentielle, sans quoi le chiffre affiché n'a aucun sens.

## Méthodologie

- **Modèle** : HGB avec les hyperparamètres optimisés de la Phase B, entraîné sur 2017-2022, évalué sur 2023 (même configuration canonique que les phases D et F).
- **Métriques** :
  - **Diagramme de fiabilité** — probabilité prédite vs taux de positifs observé, par bins. La diagonale à 45° est la calibration parfaite.
  - **Score de Brier** — erreur quadratique moyenne entre probabilité prédite et label vrai. Plus bas, mieux c'est. Se décompose en fiabilité (erreur de calibration) + résolution (pouvoir discriminant) + incertitude (erreur due au seul taux de base).
  - **Expected Calibration Error (ECE)** — moyenne |prédit − observé| pondérée par la taille des bins. Le scalaire principal de calibration.
- **Méthodes de calibration comparées** :
  - **Non calibré** (sortie HGB brute) — référence.
  - **Recalibration Platt (sigmoïde)** — ajuste une régression logistique 1D sur les prédictions mises de côté. Idéal quand la mauvaise calibration est monotone.
  - **Recalibration isotonique** — non paramétrique, ajuste n'importe quelle correspondance monotone. Meilleur quand la mauvaise calibration est non linéaire, mais demande plus de données.

**Découpage train / ajustement / évaluation pour la calibration :** pour éviter l'optimisme en intra-échantillon, on entraîne le modèle sur 2017-2022, on recalibre les probabilités via une *année de calibration* mise de côté (2022, l'année d'entraînement la plus récente — même approche que le walk-forward de la Phase B), et on évalue la calibration sur 2023.

## Ce que produit ce notebook

Sous `ml-artifacts/calibration_phase_e/` :
- `reliability_uncalibrated.png`, `reliability_platt.png`, `reliability_isotonic.png` — diagrammes de fiabilité.
- `calibration_metrics.csv` — Brier / ECE / log-loss pour chacune des trois variantes.
- `calibration_summary.md` — paragraphe prêt pour le mémoire + recommandation.

## 1. Environnement d'exécution et constantes

**Environnement CPU.** Total ~10-12 min : ~8 min pour l'entraînement du modèle, le reste pour les calculs de calibration (rapides) et les graphiques.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/zribi1/pfein.git'
BRANCH = 'data-extraction'
REPO_DIR = '/content/pfein'
BACKEND_DIR = f'{REPO_DIR}/back_end'

DRIVE_ROOT = '/content/drive/MyDrive/PFE ML Data/pfe_data'
DATA_LAKE = f'{DRIVE_ROOT}/data-lake'
ARTIFACTS_DIR = f'{DRIVE_ROOT}/ml-artifacts'
DUCKDB_TMP = '/content/pfein_duckdb_tmp'

TARGET = 'continuity_risk_12m_label'
TRAIN_MAX_ROWS = 2_000_000
START_YEAR = 2017
CALIB_YEAR = 2022       # held-out for fitting recalibrators
TEST_YEAR = 2023        # final evaluation year (Phase C canonical)
N_BINS = 15             # reliability-diagram bin count

FEATURES_GLOB = f'{DATA_LAKE}/features/company_year_features/**/*.parquet'
LABELS_GLOB = f'{DATA_LAKE}/features/risk_labels/**/*.parquet'

TUNED_HGB_PARAMS = Path(ARTIFACTS_DIR) / 'tuned_params_hgb.json'
PHASE_E_DIR = Path(ARTIFACTS_DIR) / 'calibration_phase_e'
PHASE_E_DIR.mkdir(parents=True, exist_ok=True)

Path(DUCKDB_TMP).mkdir(parents=True, exist_ok=True)

print('CALIB_YEAR =', CALIB_YEAR)
print('TEST_YEAR  =', TEST_YEAR)
print('OUT_DIR    =', PHASE_E_DIR)

## 2. Mise à jour du code et installation des dépendances

In [ ]:
import os

if not Path(REPO_DIR).exists():
    !git clone --branch "$BRANCH" "$REPO_URL" "$REPO_DIR"

%cd $REPO_DIR
!git fetch origin
!git switch "$BRANCH" || git switch -c "$BRANCH" "origin/$BRANCH"
!git pull --ff-only origin "$BRANCH"
%cd $BACKEND_DIR

os.environ['DUCKDB_TEMP_DIRECTORY'] = DUCKDB_TMP
!pip install -q -r collabs/requirements-colab.txt

In [ ]:
import json

if not TUNED_HGB_PARAMS.exists():
    raise SystemExit(
        f'Phase B output missing: {TUNED_HGB_PARAMS}.\n'
        f'Run Phase B (collabs/03_phaseB_optimisation_hyperparametres.ipynb) first.'
    )
tuned_params = json.loads(TUNED_HGB_PARAMS.read_text(encoding='utf-8'))
print('HGB tuned params:')
for k, v in tuned_params.items():
    print(f'  {k}: {v}')

## 3. Chargement des données (échantillon hash 2 M, années ≤ 2023) et construction de trois tranches

- **Train** = années `<= CALIB_YEAR - 1` (2017-2021). Entraîne le modèle.
- **Calib** = année `CALIB_YEAR` (2022). Mise de côté de l'entraînement ; utilisée pour ajuster les recalibrateurs Platt / Isotonique.
- **Test**  = année `TEST_YEAR` (2023). Mise de côté de tout ; évaluation finale.

In [ ]:
import math, duckdb, pandas as pd, numpy as np
from app.tools.train_continuity_model import EXCLUDE_COLUMNS

filters = [
    f'l."{TARGET}" IS NOT NULL',
    f'f.prediction_year >= {START_YEAR}',
    f'f.prediction_year <= {TEST_YEAR}',
]
where_sql = ' AND '.join(filters)

con = duckdb.connect()
total_rows = con.execute(f"""
    SELECT COUNT(*) FROM read_parquet('{FEATURES_GLOB}', union_by_name=true) f
    JOIN read_parquet('{LABELS_GLOB}', union_by_name=true) l USING (siren, prediction_year)
    WHERE {where_sql}
""").fetchone()[0]
print(f'Eligible rows (≤ {TEST_YEAR}): {total_rows:,}')

modulus = 1_000_000
threshold = max(1, min(modulus, math.ceil((TRAIN_MAX_ROWS / total_rows) * modulus * 1.15)))
row_hash = "hash(CAST(f.siren AS VARCHAR) || ':' || CAST(f.prediction_year AS VARCHAR))"
feature_cols = con.execute(
    f"DESCRIBE SELECT * FROM read_parquet('{FEATURES_GLOB}', union_by_name=true)"
).df()['column_name'].tolist()
select_cols = [c for c in feature_cols if c not in EXCLUDE_COLUMNS]
select_sql = ', '.join(f'f."{c}"' for c in select_cols)

df = con.execute(f"""
    SELECT {select_sql}, l."{TARGET}"
    FROM read_parquet('{FEATURES_GLOB}', union_by_name=true) f
    JOIN read_parquet('{LABELS_GLOB}', union_by_name=true) l USING (siren, prediction_year)
    WHERE {where_sql} AND {row_hash} % {modulus} < {threshold}
    ORDER BY {row_hash}
    LIMIT {TRAIN_MAX_ROWS}
""").df()
con.close()

print(f'Loaded shape: {df.shape}')
print(f'Years: {sorted(df["prediction_year"].unique())}')

In [ ]:
y = df[TARGET].astype(int)
feature_columns = [c for c in df.columns if c not in EXCLUDE_COLUMNS and c != TARGET]
X = df[feature_columns].copy()
for col in X.columns:
    if pd.api.types.is_bool_dtype(X[col]):
        X[col] = X[col].astype(float)

numeric_columns = [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c])]
categorical_columns = [c for c in X.columns if c not in numeric_columns]

train_mask = X['prediction_year'] < CALIB_YEAR
calib_mask = X['prediction_year'] == CALIB_YEAR
test_mask  = X['prediction_year'] == TEST_YEAR

X_train = X[train_mask].reset_index(drop=True); y_train = y[train_mask].reset_index(drop=True)
X_calib = X[calib_mask].reset_index(drop=True); y_calib = y[calib_mask].reset_index(drop=True)
X_test  = X[test_mask].reset_index(drop=True);  y_test  = y[test_mask].reset_index(drop=True)

print(f'Train (years < {CALIB_YEAR}): {len(X_train):,} rows, {y_train.sum():,} positives')
print(f'Calib (year = {CALIB_YEAR}):  {len(X_calib):,} rows, {y_calib.sum():,} positives')
print(f'Test  (year = {TEST_YEAR}):   {len(X_test):,} rows, {y_test.sum():,} positives')

## 4. Entraînement du HGB optimisé sur les années < 2022, score sur Calib et Test

In [ ]:
import importlib, time, app.tools.train_continuity_model as _tcm
importlib.reload(_tcm)
from app.tools.train_continuity_model import _build_model_pipeline

pipeline = _build_model_pipeline(
    family='hgb',
    categorical_columns=categorical_columns,
    train_positive_count=int((y_train == 1).sum()),
    train_negative_count=int((y_train == 0).sum()),
    extra_params=tuned_params,
)

print(f'Fitting tuned HGB on years 2017–{CALIB_YEAR-1}...')
start = time.time()
pipeline.fit(X_train, y_train)
print(f'Done in {(time.time()-start)/60:.1f} min')

from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss, log_loss
p_calib = pipeline.predict_proba(X_calib)[:, 1]
p_test_raw = pipeline.predict_proba(X_test)[:, 1]

print(f'\nUncalibrated test ranking: AP={average_precision_score(y_test, p_test_raw):.4f}, '
      f'AUC={roc_auc_score(y_test, p_test_raw):.4f}')

## 5. Ajustement de deux recalibrateurs sur la tranche 2022

Les deux recalibrateurs apprennent une correspondance 1D `probabilité_brute → probabilité_calibrée` sur le jeu mis de côté 2022, puis on applique cette correspondance aux prédictions brutes 2023.

**Platt** = ajustement sigmoïde (régression logistique sur les probabilités brutes). Idéal quand la mauvaise calibration est une simple courbe en S monotone.

**Isotonique** = ajustement par fonction en escalier monotone. Flexible, capte les mauvaises calibrations non monotones mais monotones en rang. Risque : peut sur-ajuster avec un petit jeu de calibration, mais 2022 a ~14 k positifs, c'est largement suffisant.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression

# Platt : régression logistique 1D sur la probabilité brute comme feature
platt = LogisticRegression(C=1e6, solver='lbfgs')
platt.fit(p_calib.reshape(-1, 1), y_calib)
p_test_platt = platt.predict_proba(p_test_raw.reshape(-1, 1))[:, 1]

# Isotonique : correspondance monotone non paramétrique
iso = IsotonicRegression(out_of_bounds='clip')
iso.fit(p_calib, y_calib)
p_test_iso = iso.predict(p_test_raw)

print('Recalibrators fitted on 2022 calibration slice.')
print(f'Test predictions ready: raw, Platt, Isotonic.')

## 6. Calcul de Brier, ECE, log-loss sur les trois variantes

In [ ]:
def expected_calibration_error(y_true, p_pred, n_bins=N_BINS):
    """Moyenne pondérée de | prédit − observé | sur des bins de probabilité de largeur égale."""
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_idx = np.clip(np.digitize(p_pred, bins) - 1, 0, n_bins - 1)
    ece = 0.0
    total = len(p_pred)
    for b in range(n_bins):
        mask = bin_idx == b
        n = int(mask.sum())
        if n == 0:
            continue
        pred_mean = p_pred[mask].mean()
        obs_mean = y_true.iloc[mask].mean() if hasattr(y_true, 'iloc') else y_true[mask].mean()
        ece += (n / total) * abs(pred_mean - obs_mean)
    return ece

variants = {
    'uncalibrated': p_test_raw,
    'platt': p_test_platt,
    'isotonic': p_test_iso,
}

rows = []
for name, p in variants.items():
    rows.append({
        'variant': name,
        'brier': brier_score_loss(y_test, p),
        'ece': expected_calibration_error(y_test, p, n_bins=N_BINS),
        'log_loss': log_loss(y_test, np.clip(p, 1e-7, 1 - 1e-7)),
        'mean_predicted': float(np.mean(p)),
        'mean_observed': float(y_test.mean()),
        'ap': average_precision_score(y_test, p),
        'auc': roc_auc_score(y_test, p),
    })
calib_metrics = pd.DataFrame(rows)
print('Calibration metrics on 2023 test set:')
print(calib_metrics.to_string(index=False))

calib_metrics.to_csv(PHASE_E_DIR / 'calibration_metrics.csv', index=False)
print(f'\nSaved: {PHASE_E_DIR / "calibration_metrics.csv"}')

## 7. Diagrammes de fiabilité

Chaque diagramme regroupe la probabilité prédite en `N_BINS` bins de largeur égale et trace `(prédit moyen, observé moyen)` par bin. La diagonale pointillée à 45° est la calibration parfaite. Barres en dessous = sur-confiant (prédit plus haut que la réalité) ; au-dessus = sous-confiant.

In [ ]:
import matplotlib.pyplot as plt

def reliability_curve(y_true, p_pred, n_bins=N_BINS):
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_idx = np.clip(np.digitize(p_pred, bins) - 1, 0, n_bins - 1)
    rows = []
    for b in range(n_bins):
        mask = bin_idx == b
        n = int(mask.sum())
        if n == 0:
            continue
        rows.append({
            'bin_low': bins[b],
            'bin_high': bins[b + 1],
            'n': n,
            'mean_pred': float(p_pred[mask].mean()),
            'mean_obs': float(y_true.iloc[mask].mean() if hasattr(y_true, 'iloc') else y_true[mask].mean()),
        })
    return pd.DataFrame(rows)

for name, p in variants.items():
    curve = reliability_curve(y_test, p)
    fig, (ax_top, ax_bot) = plt.subplots(
        2, 1, figsize=(7, 6.5), sharex=True,
        gridspec_kw={'height_ratios': [3, 1]},
    )
    ax_top.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='perfect calibration')
    ax_top.plot(curve['mean_pred'], curve['mean_obs'], 'o-', color='#0f766e',
                linewidth=2, markersize=7, label=name)
    ax_top.set_ylabel('Mean observed positive rate')
    ax_top.set_title(f'Phase E — Reliability diagram ({name})')
    ax_top.legend()
    ax_top.grid(True, alpha=0.3)
    ax_top.set_xlim(0, 1)
    ax_top.set_ylim(0, 1)
    ax_bot.bar(curve['mean_pred'], curve['n'], width=1 / N_BINS * 0.9,
               color='#94a3b8', alpha=0.7)
    ax_bot.set_yscale('log')
    ax_bot.set_xlabel('Mean predicted probability')
    ax_bot.set_ylabel('rows per bin (log)')
    ax_bot.grid(True, alpha=0.3)
    fig.tight_layout()
    out = PHASE_E_DIR / f'reliability_{name}.png'
    fig.savefig(out, dpi=160)
    plt.show()
    print(f'Saved: {out}')

## 8. Choix du gagnant et rédaction d'un résumé

In [ ]:
best_by_brier = calib_metrics.loc[calib_metrics['brier'].idxmin(), 'variant']
best_by_ece = calib_metrics.loc[calib_metrics['ece'].idxmin(), 'variant']

raw_row = calib_metrics[calib_metrics['variant'] == 'uncalibrated'].iloc[0]
best_row = calib_metrics[calib_metrics['variant'] == best_by_brier].iloc[0]
ece_drop = raw_row['ece'] - best_row['ece']
brier_drop = raw_row['brier'] - best_row['brier']

decision = ''
if best_by_brier == 'uncalibrated':
    decision = (
        'The uncalibrated HGB output is already the best-calibrated variant — '
        'no recalibration is needed. Ship the raw predict_proba output.'
    )
elif raw_row['ece'] < 0.02:
    decision = (
        f'The uncalibrated ECE is already {raw_row["ece"]:.4f} (< 0.02 threshold for "good enough"). '
        f'Recalibration via {best_by_brier} would reduce ECE by {ece_drop:.4f} (~{100*ece_drop/raw_row["ece"]:.0f}%), '
        f'but the gain is small. Ship the raw output unless the product is highly threshold-sensitive.'
    )
else:
    decision = (
        f'The uncalibrated ECE is {raw_row["ece"]:.4f}, which is materially miscalibrated. '
        f'Recalibrate using **{best_by_brier}** (Brier {best_row["brier"]:.4f}, ECE {best_row["ece"]:.4f} — '
        f'reduction of {brier_drop:.4f} / {ece_drop:.4f} respectively). '
        f'Save the calibrator alongside the model and apply it before exposing probabilites_cessation to users.'
    )

summary_md = (
    '# Phase E — Probability Calibration Summary\n\n'
    f'Model: tuned HGB, trained 2017–{CALIB_YEAR-1}, recalibrated on {CALIB_YEAR}, evaluated on {TEST_YEAR}.\n\n'
    f'## Metrics on 2023 test set\n\n'
    f'{calib_metrics.to_markdown(index=False)}\n\n'
    f'## Verdict\n\n'
    f'- **Best Brier**: {best_by_brier}\n'
    f'- **Best ECE**: {best_by_ece}\n'
    f'- ECE drop from recalibration: {ece_drop:+.4f}\n'
    f'- Brier drop from recalibration: {brier_drop:+.4f}\n\n'
    f'**Recommendation:** {decision}\n\n'
    f'## Note on ranking\n\n'
    f'AP and AUC are unchanged across uncalibrated / Platt / Isotonic. Both recalibrators are monotone, '
    f'so the *order* of companies by score is identical and Phase F\'s segment-robustness results carry over.\n'
)
summary_path = PHASE_E_DIR / 'calibration_summary.md'
summary_path.write_text(summary_md, encoding='utf-8')
print(summary_md)
print(f'\nSaved: {summary_path}')

## 9. Comment lire le diagramme de fiabilité

- **Points sur la diagonale à 45°** → parfaitement calibré. Quand le modèle dit `p=0,7`, 70 % de ces entreprises ont effectivement cessé.
- **Points en dessous de la diagonale à 45°** → sur-confiant. Le modèle dit `p=0,7` mais seules 50 % ont cessé. Fréquent avec l'entraînement à pondération de classe (la fonction de perte pousse les prédictions vers la classe minoritaire).
- **Points au-dessus de la diagonale à 45°** → sous-confiant. Le modèle dit `p=0,3` mais 50 % ont cessé. Moins fréquent ; généralement signe d'un signal insuffisant.
- **Le panneau du bas (compte de lignes en log)** montre où vit la majorité de la population. Un modèle peut être mal calibré uniquement dans les bins à haute probabilité (positifs rares) tout en étant bien calibré dans les bins denses à basse probabilité (la majeure partie de la population) — ce qui est souvent acceptable si le produit fait surtout remonter le haut de l'échelle de confiance.

## Paragraphe pour le mémoire

*« La Phase E a évalué la calibration des probabilités sur le jeu de test 2023, motivée par le fait que le produit présente `probabilite_cessation` à l'utilisateur sous forme de probabilité numérique plutôt que de rang. La sortie HGB non calibrée a obtenu Brier X,XXX et ECE Y,YYY. La recalibration sur un jeu mis de côté 2022 par {méthode choisie} a réduit l'ECE à Z,ZZZ (Brier W,WWW), rétablissant l'interprétation selon laquelle p = 0,7 correspond approximativement à un taux observé de positifs de 70 %. Comme les recalibrateurs Platt et Isotonique sont tous deux monotones, AP et AUC restent inchangés par rapport aux résultats des Phases B/C. »*

Compléter avec les chiffres réels des sections 6 et 8 lors du transfert vers le mémoire.

## Et après

Si la recommandation est de déployer un recalibrateur : sauvegarder `isotonic.joblib` (ou `platt.joblib`) à côté de `model.joblib` dans `ml-artifacts/`, mettre à jour `MLRegistry.load()` pour charger les deux, et ajouter une méthode `predict_proba_calibrated()` au service qui les enchaîne. Ensuite la **Phase G (latence)** pourra mesurer le pipeline d'inférence complet incluant la recalibration.